# Stock Price Forecasting
This example forecasts the price of Kweichow Moutai to show how GRU and TCN can model stock prices.

This is only a toy model and is not investment advice — markets carry risk.

## 1. Loading and Organizing the Data
Daily data downloaded from the RESSET database.

### 1.1 Loading the .xlsx data

In [ ]:
import pandas as pd

# read the Excel file
file_path = 'data/Stock/RESSET_DRESSTK_2021__1.xlsx'  # replace with your own file path
data = pd.ExcelFile(file_path)

# load a specific sheet
sheet_name = 'DRESSTK_2021_'
df = data.parse(sheet_name)


### 1.2 Align dates to Moutai's trading calendar (from the first trading day of 2021)

In [ ]:
# convert the "日期_Date" column to datetime
df['日期_Date'] = pd.to_datetime(df['日期_Date'])

# use Kweichow Moutai as the reference stock (code 600519)
guizhou_maotai = df[df['股票代码_Stkcd'] == 600809].sort_values('日期_Date')

# all trading days (based on Moutai's dates)
all_dates = guizhou_maotai['日期_Date'].unique()

# align each stock to those dates, forward-filling missing days
aligned_data = []
for stkcd, group in df.groupby('股票代码_Stkcd'):  # group by stock code
    # reindex by date using Moutai's trading calendar
    group = group.set_index('日期_Date').reindex(all_dates).sort_index()
    # forward-fill missing values (method='ffill')
    group.fillna(method='ffill', inplace=True)
    # re-add the stock-code column so it isn't lost after filling
    group['股票代码_Stkcd'] = stkcd
    # append the processed stock data to the list
    aligned_data.append(group)

# concatenate all stocks
aligned_df = pd.concat(aligned_data).reset_index().rename(columns={'index': '日期_Date'})

In [ ]:
aligned_df.describe()

### 1.3 Standardization

In [ ]:
features = ['前收盘价_PrevClPr', '开盘价_Oppr', '最高价_Hipr', '最低价_Lopr', 
            '收盘价_Clpr', '复权价1(元)_AdjClpr1', '复权价2(元)_AdjClpr2', 
            '成交量_Trdvol', '成交金额_Trdsum', '日振幅(%)_Dampltd', 
            '总股数日换手率(%)_DFulTurnR', '流通股日换手率(%)_DTrdTurnR']

# empty DataFrame to hold the standardized results
aligned_df_standardized = pd.DataFrame()

# 2. group by stock code and standardize each group
for stkcd, group in aligned_df.groupby('股票代码_Stkcd'):
    # sort by date
    group = group.sort_values('日期_Date')

    # cutoff at the first 80% of dates
    cutoff_date = group['日期_Date'].iloc[int(len(group) * 0.8)]

    # use the first 80% of dates to compute mean and std
    train_data = group[group['日期_Date'] <= cutoff_date]

    # mean and std for each feature
    mean = train_data[features].mean()
    std = train_data[features].std()

    # standardize the whole group
    group_standardized = group.copy()
    group_standardized[features] = (group[features] - mean) / (std + 1e-8)  # avoid division by zero

    # append the standardized data to the result DataFrame
    aligned_df_standardized = pd.concat([aligned_df_standardized, group_standardized], ignore_index=True)

# sort the result by date and stock code
aligned_df_standardized = aligned_df_standardized.sort_values(['日期_Date', '股票代码_Stkcd']).reset_index(drop=True)

# check the standardized result
print("Standardization complete.")
aligned_df_standardized.head()

Standardization complete.


In [ ]:
aligned_df_standardized.describe()

In [ ]:
guizhou_maotai = aligned_df_standardized[aligned_df_standardized['股票代码_Stkcd'] == 600809].sort_values('日期_Date')
guizhou_maotai.describe()

### 1.4 Building the training data and labels

In [ ]:
# feature columns used for training
import numpy as np
features = ['前收盘价_PrevClPr', '开盘价_Oppr', '最高价_Hipr', '最低价_Lopr', 
            '收盘价_Clpr', '复权价1(元)_AdjClpr1', '复权价2(元)_AdjClpr2', 
            '成交量_Trdvol', '成交金额_Trdsum', '日振幅(%)_Dampltd', 
            '总股数日换手率(%)_DFulTurnR', '流通股日换手率(%)_DTrdTurnR']

# list to hold per-trading-day feature data
stock_list = df['股票代码_Stkcd'].unique()  # all stock codes
tensor_data = []
for date in all_dates:
    day_data = aligned_df_standardized[aligned_df_standardized['日期_Date'] == date]  # data for the current day
    day_features = []  # features of every stock for this day
    for stkcd in stock_list:
        # pull this stock's features for the day; report if missing
        stock_row = day_data[day_data['股票代码_Stkcd'] == stkcd]
        if stock_row.empty:
            print(stkcd,date)
        day_features.append(stock_row[features].values.flatten())
    tensor_data.append(day_features)  # append this day's features to the list

# convert to a NumPy array of shape [days, num_stocks, feature_dim]
tensor_data = np.array(tensor_data)

# build Moutai's label tensor [days]
# build Moutai's label tensor [days]
maotai_prices = guizhou_maotai['收盘价_Clpr'].values  # Moutai's closing prices

# label: next trading day's closing price
# shift by one with np.roll; the last day has no label and is dropped
labels = np.roll(maotai_prices, -1)[:-1]  # roll by one and drop the last element
tensor_data = tensor_data[:-1]            # keep features and labels the same length

In [ ]:
tensor_data.shape

(912, 48, 12)

In [ ]:
labels.shape

(912,)

In [ ]:
labels

array([ 1.59946570e+00,  1.75623500e+00,  2.23727364e+00,  2.04110243e+00,
        1.65228103e+00,  1.81927806e+00,  1.78976854e+00,  1.65898773e+00,
        1.36406025e+00,  1.39574944e+00,  1.13586450e+00,  1.18616481e+00,
        1.47119991e+00,  1.41352222e+00,  1.69067693e+00,  1.29514882e+00,
        1.21969835e+00,  1.13586450e+00,  1.41117487e+00,  1.45778649e+00,
        1.89036916e+00,  1.90780660e+00,  1.69855731e+00,  1.77300177e+00,
        1.75623500e+00,  1.89405785e+00,  2.57009403e+00,  2.77900798e+00,
        2.23928565e+00,  1.52870993e+00,  1.37059929e+00,  9.43884985e-01,
        7.83762329e-01,  3.83036521e-01,  3.39778254e-01,  3.11274745e-01,
        4.40378876e-01,  1.92230676e-01,  3.44137615e-01, -1.76973604e-01,
       -5.42489194e-01, -1.10074190e-01,  1.79990934e-01,  3.78006490e-01,
       -7.70436532e-02,  1.97428375e-01,  2.78244208e-01,  5.34105121e-01,
        3.81359844e-01,  4.28306801e-01,  4.13384375e-01,  4.65193695e-01,
        4.78607112e-01,  

### 1.5 Building a rolling-window dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# custom Dataset
class StockDataset(Dataset):
    def __init__(self, tensor_data, labels, window_size=20):
        """
        Initialize the dataset.
        :param tensor_data: feature data of shape [days, num_stocks, feature_dim]
        :param labels: label data of shape [days]
        :param window_size: sliding-window size (e.g. 30 days)
        """
        self.data = tensor_data
        self.labels = labels
        self.window_size = window_size

    def __len__(self):
        """
        Dataset length: total days minus the window size.
        """
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        """
        Return a single sample.
        :param idx: sample index
        :return: (feature tensor, label)
        """
        x = self.data[idx:idx + self.window_size]  # the past window_size days
        y = self.labels[idx + self.window_size]  # the corresponding label
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# create the Dataset and DataLoader
window_size = 20  # sliding-window size
batch_size = 64  # samples per batch

dataset = StockDataset(tensor_data, labels, window_size=window_size)


In [ ]:
torch.save({
    'tensor_data': dataset.data,
    'labels': dataset.labels,
    'window_size': dataset.window_size
}, 'data/Stock/stock_MT.pt')

In [ ]:
# load the saved data
saved_data = torch.load('data/Stock/stock_MT.pt',weights_only=False)

# rebuild the StockDataset object
loaded_dataset = StockDataset(
    tensor_data=saved_data['tensor_data'],
    labels=saved_data['labels'],
    window_size=saved_data['window_size']
)

print("Dataset loaded")

### 1.6 Building train / validation / test sets

In [ ]:
# total dataset length
total_samples = len(dataset)

# split indices
train_end = int(total_samples * 0.8)  # end of the train set
val_end = int(total_samples * 0.9)    # end of the validation set

# split sequentially (no shuffling)
train_data = torch.utils.data.Subset(dataset, range(0, train_end))
val_data = torch.utils.data.Subset(dataset, range(train_end, val_end))
test_data = torch.utils.data.Subset(dataset, range(val_end, total_samples))

In [ ]:
# DataLoader settings
batch_size = 32  # samples per batch

# create DataLoaders, keeping order
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# check the sizes of the train/val/test sets
print(f"Train set size: {len(train_data)} samples")
print(f"Validation set size: {len(val_data)} samples")
print(f"Test set size: {len(test_data)} samples")

# inspect the DataLoader output
for batch_idx, (x, y) in enumerate(train_loader):
    print(f"Train set - Batch {batch_idx}")
    print(f"Features shape: {x.shape}")  # [batch_size, window_size, num_stocks, feature_dim]
    print(f"Labels shape: {y.shape}")    # [batch_size]
    break  # one batch is enough to check

## 2. GRU Implementation

In [ ]:
import torch
import torch.nn as nn

class MultiLayerGRU(nn.Module):
    def __init__(self, stock_count, feature_dim, hidden_dim, num_layers, output_dim):
        """
        Multi-layer GRU model.
        :param stock_count: number of stocks
        :param feature_dim: feature dimension per stock
        :param hidden_dim: GRU hidden dimension
        :param num_layers: number of GRU layers
        :param output_dim: output dimension (the predicted price)
        """
        super(MultiLayerGRU, self).__init__()
        
        # GRU input dimension
        self.input_dim = stock_count * feature_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # GRU layers
        self.gru = nn.GRU(
            input_size=self.input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True
        )
        
        # fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        """
        Forward pass.
        :param x: input of shape [batch_size, window_size, num_stocks, feature_dim]
        :return: predictions of shape [batch_size, output_dim]
        """
        batch_size, window_size, stock_count, feature_dim = x.shape
        
        # merge num_stocks and feature_dim
        x = x.view(batch_size, window_size, stock_count * feature_dim)  # [batch_size, window_size, input_dim]
        
        # initialize the hidden state of shape [num_layers, batch_size, hidden_dim]
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        
        # GRU forward pass
        out, _ = self.gru(x, h0)  # out: [batch_size, window_size, hidden_dim]
        
        # take the last time step's output
        out = out[:, -1, :]  # [batch_size, hidden_dim]
        
        # fully connected layer
        out = self.fc(out)  # [batch_size, output_dim]
        return out

In [ ]:
epochs = 10  # max number of training epochs
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose device: GPU (CUDA) if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

mps


In [ ]:
# model hyperparameters
stock_count = 48       # number of stocks
feature_dim = 12       # feature dimension per stock
hidden_dim = 128        # GRU hidden dimension
num_layers = 3         # number of GRU layers
output_dim = 1         # output dimension (the predicted price)

# instantiate the model
model_GRU = MultiLayerGRU(stock_count, feature_dim, hidden_dim, num_layers, output_dim).to(device)

# print the model architecture
print(model_GRU)

MultiLayerGRU(
  (gru): GRU(576, 128, num_layers=3, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
# the train/test functions need to be adapted
# define the training function
def train(args, model, device, train_loader, optimizer, epoch):
    """
    Training loop.
    :param args: object/namespace of hyperparameters (e.g. batch_size, learning_rate).
    :param model: the neural network to be trained.
    :param device: where model and data run (cpu or cuda).
    :param train_loader: yields batches of training data.
    :param optimizer: updates model parameters (e.g. SGD, Adam).
    :param epoch: current epoch number, for progress logging.
    """
    model.train()  # switch to training mode
    for batch_idx, (data, target) in enumerate(train_loader):  # load a batch from train_loader
        data, target = data.to(device), target.to(device)  # move data and labels to the device (CPU or GPU)
        optimizer.zero_grad()  # clear all parameter gradients
        output = model(data)  # forward pass on the current batch
        loss = torch.nn.MSELoss()(output.squeeze(), target)  # compute the loss with mean squared error
        loss.backward()  # backpropagation to compute gradients
        optimizer.step()  # update model parameters with the optimizer
        

# define the test function
def test(model, device, test_loader):
    """
    Evaluation loop.
    :param model: the network to evaluate.
    :param device: where model and data run (CPU or GPU).
    :param test_loader: provides the test set (usually unshuffled).
    """
    model.eval()  # switch to eval mode
    test_loss = 0  # accumulated test loss
    all_preds = []  # store all predictions
    all_targets = []  # store all ground-truth values

    with torch.no_grad():  # disable autograd to save memory
        for data, target in test_loader:
            data, target = data.to(device), target.to(device).float()  # move data and labels to the device
            output = model(data)  # forward pass to get predictions
            test_loss += torch.nn.MSELoss()(output.squeeze(), target).item() * len(target)  # accumulate the loss
            all_preds.extend(output.squeeze().cpu().numpy())  # store predictions
            all_targets.extend(target.cpu().numpy())  # store ground-truth values

    # average loss
    test_loss /= len(test_loader.dataset)

    # compute R^2 (coefficient of determination)
    r2 = r2_score(all_targets, all_preds)

    # print the evaluation results
    print("=" * 50)
    print("Evaluation Results:")
    print(f" - Average Loss: {test_loss:.4f}")
    print(f" - R^2 Score: {r2:.4f}")
    print("=" * 50)


In [ ]:
import time
import torch.optim as optim
from sklearn.metrics import r2_score

optimizer = optim.Adadelta(model_GRU.parameters())

In [ ]:
import time
for epoch in range(1, epochs + 1):
    start_time = time.time()  # record start time
    train([], model_GRU, device, train_loader, optimizer, epoch)  # train
    train_time = time.time()
    test(model_GRU, device, val_loader)  # test
    test_time = time.time()
    train_duration = train_time - start_time
    test_duration = test_time - train_time
    print(f"Epoch {epoch}/{epochs} - Trained in {train_duration:.2f} seconds  - Test in {test_duration:.2f} seconds.\n")


Evaluation Results:
 - Average Loss: 0.2326
 - R^2 Score: -0.9298
Epoch 1/10 - Trained in 2.92 seconds  - Test in 0.15 seconds.

Evaluation Results:
 - Average Loss: 0.1226
 - R^2 Score: -0.0168
Epoch 2/10 - Trained in 0.87 seconds  - Test in 0.04 seconds.

Evaluation Results:
 - Average Loss: 0.1253
 - R^2 Score: -0.0396
Epoch 3/10 - Trained in 0.86 seconds  - Test in 0.04 seconds.

Evaluation Results:
 - Average Loss: 0.1041
 - R^2 Score: 0.1361
Epoch 4/10 - Trained in 0.88 seconds  - Test in 0.04 seconds.

Evaluation Results:
 - Average Loss: 0.1773
 - R^2 Score: -0.4708
Epoch 5/10 - Trained in 0.88 seconds  - Test in 0.05 seconds.

Evaluation Results:
 - Average Loss: 0.1464
 - R^2 Score: -0.2147
Epoch 6/10 - Trained in 0.87 seconds  - Test in 0.04 seconds.

Evaluation Results:
 - Average Loss: 0.2352
 - R^2 Score: -0.9514
Epoch 7/10 - Trained in 0.86 seconds  - Test in 0.04 seconds.

Evaluation Results:
 - Average Loss: 0.2451
 - R^2 Score: -1.0336
Epoch 8/10 - Trained in 0.85 sec

## 3. TCN

In [ ]:
import torch.nn.functional as F

class TemporalBlock(nn.Module):
    """A single TCN block: dilated convolution, activation, dropout and a residual connection."""
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, dropout):
        """
        :param in_channels: number of input channels
        :param out_channels: number of output channels
        :param kernel_size: convolution kernel size
        :param stride: stride
        :param dilation: dilation rate
        :param dropout: dropout probability
        """
        super(TemporalBlock, self).__init__()

        # left padding size
        self.padding = (kernel_size - 1) * dilation

        # first dilated convolution
        self.conv1 = nn.Conv1d(
            in_channels, out_channels, kernel_size, stride=stride, dilation=dilation
        )
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        # second dilated convolution
        self.conv2 = nn.Conv1d(
            out_channels, out_channels, kernel_size, stride=stride, dilation=dilation
        )
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        # residual connection (add a projection if channel counts differ)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        Forward pass.
        :param x: input tensor of shape [batch_size, channels, sequence_length]
        :return: output tensor of shape [batch_size, out_channels, sequence_length]
        """
        # left padding
        x_padded = F.pad(x, (self.padding, 0))  # pad only the beginning of the sequence

        # first convolution
        out = self.conv1(x_padded)
        out = self.relu1(out)
        out = self.dropout1(out)

        # second convolution
        out = F.pad(out, (self.padding, 0))  # the second convolution also needs left padding
        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        # match the residual length
        res = x if self.downsample is None else self.downsample(x)
        res = res[:, :, -out.size(2):]  # crop to match out

        return self.relu(out + res)
        
class TemporalConvNet(nn.Module):
    """The main TCN network: a stack of TemporalBlocks."""
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        """
        :param num_inputs: number of input channels
        :param num_channels: list of output-channel counts per layer
        :param kernel_size: convolution kernel size
        :param dropout: dropout probability
        """
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)  # number of layers
        for i in range(num_levels):
            dilation_size = 2 ** i  # dilation rate for this layer
            in_channels = num_inputs if i == 0 else num_channels[i-1]  # input channels
            out_channels = num_channels[i]  # output channels
            layers += [TemporalBlock(
                in_channels, out_channels, kernel_size, stride=1,
                dilation=dilation_size, dropout=dropout
            )]

        # combine all layers
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        """
        Forward pass.
        :param x: input tensor of shape [batch_size, num_inputs, sequence_length]
        :return: output tensor of shape [batch_size, num_channels[-1], sequence_length]
        """
        return self.network(x)


class StockPriceTCN(nn.Module):
    """TCN for stock-price prediction."""
    def __init__(self, num_stocks, feature_dim, num_channels, kernel_size, dropout):
        """
        :param num_stocks: number of stocks
        :param feature_dim: feature dimension per stock
        :param num_channels: list of channel counts per layer
        :param kernel_size: convolution kernel size
        :param dropout: dropout probability
        """
        super(StockPriceTCN, self).__init__()
        # the TCN input dimension is num_stocks * feature_dim
        self.tcn = TemporalConvNet(num_stocks * feature_dim, num_channels, kernel_size, dropout)
        self.fc = nn.Linear(num_channels[-1], 1)  # fully connected layer outputting a single value (the predicted price)

    def forward(self, x):
        """
        Forward pass.
        :param x: input tensor of shape [batch_size, window_size, num_stocks, feature_dim]
        :return: predictions of shape [batch_size, 1]
        """
        batch_size, window_size, num_stocks, feature_dim = x.shape

        # reshape the input to [batch_size, num_stocks * feature_dim, window_size]
        x = x.permute(0, 2, 3, 1).contiguous()  # permute to [batch_size, num_stocks, feature_dim, window_size]
        x = x.view(batch_size, num_stocks * feature_dim, window_size)  # merge num_stocks and feature_dim

        # pass through the TCN
        out = self.tcn(x)  # TCN output [batch_size, num_channels[-1], window_size]
        out = out[:, :, -1]  # take the last time step's output [batch_size, num_channels[-1]]

        # pass through the fully connected layer
        out = self.fc(out)  # map to the final prediction [batch_size, 1]
        return out

In [ ]:

# settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = 12  # feature dimension per stock
stock_count = 48       # number of stocks
num_channels = [32, 128, 512]  # channels per TCN layer
kernel_size = 3
dropout = 0.6

# model, optimizer and loss function
model_TCN = StockPriceTCN(stock_count, input_dim, num_channels, kernel_size, dropout).to(device)
#optimizer = torch.optim.Adam(model_TCN.parameters(), lr=0.001)
optimizer = optim.Adadelta(model_TCN.parameters())
criterion = nn.MSELoss()

print(model_TCN)

# training and testing
num_epochs = 20

StockPriceTCN(
  (tcn): TemporalConvNet(
    (network): Sequential(
      (0): TemporalBlock(
        (conv1): Conv1d(576, 32, kernel_size=(3,), stride=(1,))
        (relu1): ReLU()
        (dropout1): Dropout(p=0.6, inplace=False)
        (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
        (relu2): ReLU()
        (dropout2): Dropout(p=0.6, inplace=False)
        (downsample): Conv1d(576, 32, kernel_size=(1,), stride=(1,))
        (relu): ReLU()
      )
      (1): TemporalBlock(
        (conv1): Conv1d(32, 128, kernel_size=(3,), stride=(1,), dilation=(2,))
        (relu1): ReLU()
        (dropout1): Dropout(p=0.6, inplace=False)
        (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), dilation=(2,))
        (relu2): ReLU()
        (dropout2): Dropout(p=0.6, inplace=False)
        (downsample): Conv1d(32, 128, kernel_size=(1,), stride=(1,))
        (relu): ReLU()
      )
      (2): TemporalBlock(
        (conv1): Conv1d(128, 512, kernel_size=(3,), stride=(1,), dilatio

In [ ]:
import time
for epoch in range(1, epochs + 1):
    start_time = time.time()  # record start time
    train([], model_TCN, device, train_loader, optimizer, epoch)  # train
    train_time = time.time()
    test(model_TCN, device, val_loader)  # test
    test_time = time.time()
    train_duration = train_time - start_time
    test_duration = test_time - train_time
    print(f"Epoch {epoch}/{epochs} - Trained in {train_duration:.2f} seconds  - Test in {test_duration:.2f} seconds.\n")


Evaluation Results:
 - Average Loss: 0.1704
 - R^2 Score: -0.4134
Epoch 1/10 - Trained in 1.37 seconds  - Test in 0.14 seconds.

Evaluation Results:
 - Average Loss: 0.6745
 - R^2 Score: -4.5958
Epoch 2/10 - Trained in 1.33 seconds  - Test in 0.14 seconds.

Evaluation Results:
 - Average Loss: 0.2961
 - R^2 Score: -1.4568
Epoch 3/10 - Trained in 1.36 seconds  - Test in 0.14 seconds.

Evaluation Results:
 - Average Loss: 0.4553
 - R^2 Score: -2.7774
Epoch 4/10 - Trained in 1.38 seconds  - Test in 0.14 seconds.

Evaluation Results:
 - Average Loss: 0.6152
 - R^2 Score: -4.1034
Epoch 5/10 - Trained in 1.42 seconds  - Test in 0.15 seconds.

Evaluation Results:
 - Average Loss: 0.1216
 - R^2 Score: -0.0091
Epoch 6/10 - Trained in 1.46 seconds  - Test in 0.15 seconds.

Evaluation Results:
 - Average Loss: 0.6072
 - R^2 Score: -4.0370
Epoch 7/10 - Trained in 1.45 seconds  - Test in 0.15 seconds.

Evaluation Results:
 - Average Loss: 0.6702
 - R^2 Score: -4.5596
Epoch 8/10 - Trained in 1.46 se